# Workflow Revisi — REV dan Anisotropi Kirby

Notebook ini saya gunakan untuk menelusuri pertanyaan Bab 6 secara bertahap. Saya mulai dari pemeriksaan citra dan ukuran pori, menghitung sifat geometri, lalu menjalankan simulasi aliran. Subvolume yang tidak menembus dan simulasi yang belum konvergen tetap dicatat sebagai bagian dari hasil.

**Sebelum mulai:** pilih `Runtime → Change runtime type → T4 GPU` (atau GPU yang tersedia), lalu pastikan folder kerja tersimpan di Google Drive. Jalankan tahap secara berurutan. Bila runtime terputus, buka kembali notebook dan lanjutkan dari tahap yang belum selesai; checkpoint disimpan di folder proyek Google Drive.

**Catatan keterulangan:** seed, ukuran domain, koordinat subvolume, parameter solver, dan hasil tiap kasus disimpan. Saya membaca tabel dan grafik setelah setiap tahap, bukan hanya menjalankan seluruh sel sampai akhir.

**Penggunaan AI:** ChatGPT digunakan sebagai pendamping untuk merapikan dokumen, menata struktur dan penjelasan sel Google Colab, menjelaskan materi perkuliahan, serta memberi panduan saat memahami dan menjalankan tahapan pengerjaan. ChatGPT juga membantu menyusun dan meninjau beberapa bagian kode pendukung. Eksekusi program, pemeriksaan keluaran, keputusan analisis, dan penafsiran hasil tetap saya periksa dan pertanggungjawabkan.


**Penyiapan data:** citra RAW resmi tidak disertakan dalam repositori publik. Letakkan `Kirby_2d25um_binary_256.raw` dari bundel resmi mata kuliah pada `data/citra/` sebelum menjalankan analisis. Jangan mengubah isi citra atau metadata.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os, sys, time

drive.mount('/content/drive')
ROOT_DRIVE = Path('/content/drive/MyDrive/REV-Anisotropi-Kirby-Kerja')
required = [
    ROOT_DRIVE / 'program/inti/rev_anisotropi.py',
    ROOT_DRIVE / 'data/metadata.json',
    ROOT_DRIVE / 'requirements.txt',
]

# Jika struktur proyek belum tersedia, unggah ZIP repository yang sudah diekstrak dari GitHub.
if not all(p.exists() for p in required):
    print('Unggah ZIP repository REV-Anisotropi-Kirby (Code → Download ZIP dari GitHub).')
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('File ZIP proyek belum diunggah.')
    zip_name = zip_names[0]
    tmp = Path('/content/project_upload')
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(tmp)
    candidates = [p for p in tmp.rglob('*')
                  if p.is_dir() and (p / 'program/inti/rev_anisotropi.py').exists()
                  and (p / 'data/metadata.json').exists()]
    if not candidates:
        raise FileNotFoundError('Struktur repository tidak lengkap atau ZIP yang dipilih salah.')
    candidate = candidates[0]
    if ROOT_DRIVE.exists():
        stamp = time.strftime('%Y%m%d_%H%M%S')
        backup = ROOT_DRIVE.with_name(ROOT_DRIVE.name + '_backup_' + stamp)
        shutil.move(str(ROOT_DRIVE), str(backup))
        print('Folder proyek lama disimpan sebagai:', backup)
    shutil.copytree(candidate, ROOT_DRIVE)

# RAW sengaja tidak berada di repository publik. Unggah file dari bundel resmi mata kuliah.
raw_path = ROOT_DRIVE / 'data/citra/Kirby_2d25um_binary_256.raw'
if not raw_path.exists():
    print('Unggah Kirby_2d25um_binary_256.raw dari bundel resmi mata kuliah.')
    uploaded_raw = files.upload()
    raw_names = [n for n in uploaded_raw if Path(n).name == 'Kirby_2d25um_binary_256.raw']
    if not raw_names:
        raise FileNotFoundError('File Kirby_2d25um_binary_256.raw belum diunggah.')
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(raw_names[0], raw_path)

print('Folder kerja:', ROOT_DRIVE)
print('Berkas inti lengkap:', all(p.exists() for p in required))
print('Citra RAW ditemukan:', raw_path.exists(), '| ukuran:', raw_path.stat().st_size if raw_path.exists() else 0)


In [ ]:
%cd /content/drive/MyDrive/REV-Anisotropi-Kirby-Kerja
!pip -q install -r requirements.txt
!python -c "import json,numpy as np; from pathlib import Path; m=json.loads(Path('data/metadata.json').read_text()); a=np.fromfile(Path('data')/m['raw_file'],dtype=np.uint8); assert a.size==256**3; print('Raw file valid; porosity =',float((a.reshape(256,256,256)==0).mean()))"

In [ ]:
# Pemeriksaan domain acuan: ketiga takrif porositas dihitung ulang dari citra.
!python program/inti/rev_anisotropi.py --stage domain

import pandas as pd
from IPython.display import display
domain_penuh = pd.read_csv('hasil/porositas_domain_penuh.csv')
display(domain_penuh.round(6))


In [ ]:
# CuPy hanya dipasang bila runtime benar-benar menyediakan GPU NVIDIA.
import subprocess, re, shutil, sys
if shutil.which('nvidia-smi'):
    info = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
    print(info.splitlines()[0] if info else 'GPU NVIDIA terdeteksi.')
    match = re.search(r'CUDA Version: (\d+)', info)
    major = int(match.group(1)) if match else 12
    paket = 'cupy-cuda13x' if major >= 13 else 'cupy-cuda12x'
    hasil_pip = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paket])
    if hasil_pip.returncode != 0 and paket == 'cupy-cuda13x':
        print('Pemasangan CuPy CUDA 13 tidak cocok; mencoba wheel CUDA 12.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'], check=True)
    import cupy as cp
    print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())
else:
    print('GPU NVIDIA tidak terdeteksi. Backend CPU dapat dipakai untuk uji kecil, tetapi LBM lengkap sebaiknya dijalankan pada runtime GPU.')


## 1. Pemeriksaan citra dan kelayakan resolusi (Pertemuan 2)

Sebelum menghitung REV, saya periksa apakah resolusi voxel cukup untuk menangkap pori/leher pori yang mengendalikan aliran. Saya gunakan granulometri dari notebook P2 pada citra biner yang disediakan. Ini adalah indikator berbasis geometri pori, bukan pengukuran langsung setiap leher pori.

Bundel yang tersedia hanya berisi citra biner. Karena itu, saya tidak dapat menilai artefak grayscale seperti beam hardening, ring artefact, atau partial-volume dari data ini.


In [ ]:
!python program/inti/granulometri_kelayakan.py

import json
from pathlib import Path
result = json.loads(Path('hasil/granulometri_ringkasan.json').read_text())
print(f"r50 = {result['r50_voxel']:.3f} voxel ({result['r50_um']:.2f} µm)")
print(f"d_t perkiraan = {result['estimated_controlling_throat_diameter_um']:.2f} µm")
print(f"d_t / Δx = {result['resolution_ratio_d_t_over_dx']:.2f} voxel → {result['decision']}")
from IPython.display import display, Image
display(Image('gambar/granulometri_leher_pori.png'))


**Catatan saya setelah melihat hasil:** apakah rasio d_t/Δx berada pada kategori yang disarankan P2? Apa keterbatasan karena citra yang tersedia sudah biner? Tuliskan jawaban dengan angka dari output, bukan hanya menyebut citra 'bagus' atau 'jelek'.


## 2. Hitung sifat geometri

30 subvolume per ukuran untuk porositas dan luas permukaan spesifik. Hasil, koordinat, dan ringkasan REV tersimpan di `hasil/`.

In [ ]:
!python program/inti/rev_anisotropi.py --stage geometry

In [ ]:
import pandas as pd
from IPython.display import display, Image
cv_geometri = pd.read_csv('hasil/koefisien_variasi_geometri.csv')
print('Koefisien variasi (%):')
display(cv_geometri.pivot(index='L_voxel', columns='property', values='Cv').mul(100).round(2))
print('Diagnostik tumpang tindih pasangan subvolume:')
display(pd.read_csv('hasil/diagnostik_tumpang_tindih.csv'))
display(Image('gambar/cv_geometri.png'))


**Catatan saya:** kurva mana yang paling cepat turun? Pada ukuran 128 voxel, apakah nilai Cv di bawah 5% cukup untuk menyatakan REV? Periksa juga fraksi tumpang tindih di tabel; ukuran besar bisa membuat subvolume yang berbeda koordinat tetapi memakai voxel yang sama.


## 3. Verifikasi solver dengan kanal lurus

Pembanding analitik adalah solusi Poiseuille dalam kanal silindris. Catat galat relatif dan status konvergensi untuk setiap radius. Ketiga radius dipakai untuk melihat efek diskretisasi.

In [ ]:
!python program/lbm/verifikasi_analitik.py

## 4. Hitung tortuositas geometri

Menggunakan Dijkstra 6-tetangga pada satu subvolume 128³ terpilih; dua sumbu melintang periodik, sumbu aliran tidak periodik. Ini adalah tortuositas geometri, bukan tortuositas aliran.

In [ ]:
!python program/inti/tortuositas_geometrik.py

## 5. Simulasi permeabilitas subvolume (LBM D3Q19)

Saya menjalankan ukuran satu per satu supaya keluaran dan pemakaian GPU mudah diperiksa. Untuk setiap ukuran, enam subvolume diuji pada arah X, Y, dan Z. Program menyimpan koordinat, nilai permeabilitas, riwayat residual, variasi fluks antaririsan, dan pemeriksaan fisis ke checkpoint.

Satu kasus baru boleh dipakai dalam statistik jika memenuhi dua syarat konvergensi sekaligus (perubahan relatif kecepatan < 10⁻⁷ dan Cv fluks antaririsan < 10⁻³) serta pemeriksaan Mach, Reynolds pori, dan perubahan massa. Kasus tidak menembus tetap dicatat terpisah. Jalankan batch berikut berurutan; setelah tiap batch, baca ringkasannya dahulu.


### Batch L = 32 voxel (paling kecil)

Sebelum menjalankan sel, saya pastikan runtime Colab masih menggunakan GPU. Batas 400.000 langkah mengikuti rancangan eksperimen. Jika proses belum selesai, jangan jalankan ukuran lain dulu; biarkan sel menyelesaikan kasusnya atau sambung kembali dari checkpoint setelah runtime tersambung ulang.


In [ ]:
# Saya jalankan satu ukuran terlebih dahulu agar hasilnya mudah diaudit.
!python program/inti/rev_anisotropi.py --stage flow --backend auto --flow-sizes 32 --max-steps 400000


In [ ]:
# Pemeriksaan setelah batch L=32: jumlah kasus, jalur tembus, konvergensi,
# serta pemeriksaan fisis. Baris checkpoint tetap disimpan walaupun belum lolos.
import pandas as pd
from pathlib import Path
checkpoint = Path('hasil/permeabilitas_subvolume_checkpoint.csv')
if checkpoint.exists():
    cek = pd.read_csv(checkpoint)
    cek = cek[cek['L_voxel'] == 32]
    if len(cek):
        display(cek.groupby('axis').agg(
            n_tercatat=('replicate','count'),
            n_menembus=('through','sum'),
            n_konvergen=('converged','sum'),
            n_lolos_fisis=('physical_checks_json', lambda x: sum(
                all(__import__('json').loads(v or '{}').values()) if isinstance(v,str) else False for v in x
            ))
        ).reset_index())
        display(cek[['replicate','axis','k_mD','through','converged','steps','slice_flux_cv','physical_checks_json']])
    else:
        print('Belum ada keluaran checkpoint untuk L=32.')
else:
    print('Checkpoint belum dibuat.')


**Catatan saya setelah L=32:** berapa kasus yang menembus? Apakah ada kasus yang belum konvergen atau belum lolos pemeriksaan fisis? Saya tidak menghapus baris yang bermasalah. Jika ada kasus yang belum lengkap, saya selesaikan batch ini sebelum pindah ke ukuran berikutnya.


### Batch L = 48 voxel (kecil)

Sebelum menjalankan sel, saya pastikan runtime Colab masih menggunakan GPU. Batas 400.000 langkah mengikuti rancangan eksperimen. Jika proses belum selesai, jangan jalankan ukuran lain dulu; biarkan sel menyelesaikan kasusnya atau sambung kembali dari checkpoint setelah runtime tersambung ulang.


In [ ]:
# Saya jalankan satu ukuran terlebih dahulu agar hasilnya mudah diaudit.
!python program/inti/rev_anisotropi.py --stage flow --backend auto --flow-sizes 48 --max-steps 400000


In [ ]:
# Pemeriksaan setelah batch L=48: jumlah kasus, jalur tembus, konvergensi,
# serta pemeriksaan fisis. Baris checkpoint tetap disimpan walaupun belum lolos.
import pandas as pd
from pathlib import Path
checkpoint = Path('hasil/permeabilitas_subvolume_checkpoint.csv')
if checkpoint.exists():
    cek = pd.read_csv(checkpoint)
    cek = cek[cek['L_voxel'] == 48]
    if len(cek):
        display(cek.groupby('axis').agg(
            n_tercatat=('replicate','count'),
            n_menembus=('through','sum'),
            n_konvergen=('converged','sum'),
            n_lolos_fisis=('physical_checks_json', lambda x: sum(
                all(__import__('json').loads(v or '{}').values()) if isinstance(v,str) else False for v in x
            ))
        ).reset_index())
        display(cek[['replicate','axis','k_mD','through','converged','steps','slice_flux_cv','physical_checks_json']])
    else:
        print('Belum ada keluaran checkpoint untuk L=48.')
else:
    print('Checkpoint belum dibuat.')


**Catatan saya setelah L=48:** berapa kasus yang menembus? Apakah ada kasus yang belum konvergen atau belum lolos pemeriksaan fisis? Saya tidak menghapus baris yang bermasalah. Jika ada kasus yang belum lengkap, saya selesaikan batch ini sebelum pindah ke ukuran berikutnya.


### Batch L = 64 voxel (menengah)

Sebelum menjalankan sel, saya pastikan runtime Colab masih menggunakan GPU. Batas 400.000 langkah mengikuti rancangan eksperimen. Jika proses belum selesai, jangan jalankan ukuran lain dulu; biarkan sel menyelesaikan kasusnya atau sambung kembali dari checkpoint setelah runtime tersambung ulang.


In [ ]:
# Saya jalankan satu ukuran terlebih dahulu agar hasilnya mudah diaudit.
!python program/inti/rev_anisotropi.py --stage flow --backend auto --flow-sizes 64 --max-steps 400000


In [ ]:
# Pemeriksaan setelah batch L=64: jumlah kasus, jalur tembus, konvergensi,
# serta pemeriksaan fisis. Baris checkpoint tetap disimpan walaupun belum lolos.
import pandas as pd
from pathlib import Path
checkpoint = Path('hasil/permeabilitas_subvolume_checkpoint.csv')
if checkpoint.exists():
    cek = pd.read_csv(checkpoint)
    cek = cek[cek['L_voxel'] == 64]
    if len(cek):
        display(cek.groupby('axis').agg(
            n_tercatat=('replicate','count'),
            n_menembus=('through','sum'),
            n_konvergen=('converged','sum'),
            n_lolos_fisis=('physical_checks_json', lambda x: sum(
                all(__import__('json').loads(v or '{}').values()) if isinstance(v,str) else False for v in x
            ))
        ).reset_index())
        display(cek[['replicate','axis','k_mD','through','converged','steps','slice_flux_cv','physical_checks_json']])
    else:
        print('Belum ada keluaran checkpoint untuk L=64.')
else:
    print('Checkpoint belum dibuat.')


**Catatan saya setelah L=64:** berapa kasus yang menembus? Apakah ada kasus yang belum konvergen atau belum lolos pemeriksaan fisis? Saya tidak menghapus baris yang bermasalah. Jika ada kasus yang belum lengkap, saya selesaikan batch ini sebelum pindah ke ukuran berikutnya.


### Batch L = 96 voxel (besar)

Sebelum menjalankan sel, saya pastikan runtime Colab masih menggunakan GPU. Batas 400.000 langkah mengikuti rancangan eksperimen. Jika proses belum selesai, jangan jalankan ukuran lain dulu; biarkan sel menyelesaikan kasusnya atau sambung kembali dari checkpoint setelah runtime tersambung ulang.


In [ ]:
# Saya jalankan satu ukuran terlebih dahulu agar hasilnya mudah diaudit.
!python program/inti/rev_anisotropi.py --stage flow --backend auto --flow-sizes 96 --max-steps 400000


In [ ]:
# Pemeriksaan setelah batch L=96: jumlah kasus, jalur tembus, konvergensi,
# serta pemeriksaan fisis. Baris checkpoint tetap disimpan walaupun belum lolos.
import pandas as pd
from pathlib import Path
checkpoint = Path('hasil/permeabilitas_subvolume_checkpoint.csv')
if checkpoint.exists():
    cek = pd.read_csv(checkpoint)
    cek = cek[cek['L_voxel'] == 96]
    if len(cek):
        display(cek.groupby('axis').agg(
            n_tercatat=('replicate','count'),
            n_menembus=('through','sum'),
            n_konvergen=('converged','sum'),
            n_lolos_fisis=('physical_checks_json', lambda x: sum(
                all(__import__('json').loads(v or '{}').values()) if isinstance(v,str) else False for v in x
            ))
        ).reset_index())
        display(cek[['replicate','axis','k_mD','through','converged','steps','slice_flux_cv','physical_checks_json']])
    else:
        print('Belum ada keluaran checkpoint untuk L=96.')
else:
    print('Checkpoint belum dibuat.')


**Catatan saya setelah L=96:** berapa kasus yang menembus? Apakah ada kasus yang belum konvergen atau belum lolos pemeriksaan fisis? Saya tidak menghapus baris yang bermasalah. Jika ada kasus yang belum lengkap, saya selesaikan batch ini sebelum pindah ke ukuran berikutnya.


### Batch L = 128 voxel (terbesar)

Sebelum menjalankan sel, saya pastikan runtime Colab masih menggunakan GPU. Batas 400.000 langkah mengikuti rancangan eksperimen. Jika proses belum selesai, jangan jalankan ukuran lain dulu; biarkan sel menyelesaikan kasusnya atau sambung kembali dari checkpoint setelah runtime tersambung ulang.


In [ ]:
# Saya jalankan satu ukuran terlebih dahulu agar hasilnya mudah diaudit.
!python program/inti/rev_anisotropi.py --stage flow --backend auto --flow-sizes 128 --max-steps 400000


In [ ]:
# Pemeriksaan setelah batch L=128: jumlah kasus, jalur tembus, konvergensi,
# serta pemeriksaan fisis. Baris checkpoint tetap disimpan walaupun belum lolos.
import pandas as pd
from pathlib import Path
checkpoint = Path('hasil/permeabilitas_subvolume_checkpoint.csv')
if checkpoint.exists():
    cek = pd.read_csv(checkpoint)
    cek = cek[cek['L_voxel'] == 128]
    if len(cek):
        display(cek.groupby('axis').agg(
            n_tercatat=('replicate','count'),
            n_menembus=('through','sum'),
            n_konvergen=('converged','sum'),
            n_lolos_fisis=('physical_checks_json', lambda x: sum(
                all(__import__('json').loads(v or '{}').values()) if isinstance(v,str) else False for v in x
            ))
        ).reset_index())
        display(cek[['replicate','axis','k_mD','through','converged','steps','slice_flux_cv','physical_checks_json']])
    else:
        print('Belum ada keluaran checkpoint untuk L=128.')
else:
    print('Checkpoint belum dibuat.')


**Catatan saya setelah L=128:** berapa kasus yang menembus? Apakah ada kasus yang belum konvergen atau belum lolos pemeriksaan fisis? Saya tidak menghapus baris yang bermasalah. Jika ada kasus yang belum lengkap, saya selesaikan batch ini sebelum pindah ke ukuran berikutnya.


### Ringkasan penyapuan LBM

Sesudah lima ukuran selesai, saya memeriksa apakah ada 90 kombinasi ukuran–replikasi–arah yang tercatat. Jumlah kasus yang layak untuk statistik dapat lebih sedikit, tetapi semua kasus harus tetap muncul dalam file mentah.


In [ ]:
import pandas as pd
from pathlib import Path
flow_path = Path('hasil/permeabilitas_subvolume.csv')
if flow_path.exists():
    flow = pd.read_csv(flow_path)
    print('Jumlah baris:', len(flow), 'dari maksimum 90')
    display(flow.groupby(['L_voxel','axis']).agg(
        n_cases=('replicate','count'),
        n_through=('through','sum'),
        n_converged=('converged','sum'),
        n_lolos_fisis=('physical_checks_json', lambda x: sum(
            all(__import__('json').loads(v or '{}').values()) if isinstance(v,str) else False for v in x
        ))
    ).reset_index())
    if Path('hasil/koefisien_variasi_permeabilitas.csv').exists():
        display(pd.read_csv('hasil/koefisien_variasi_permeabilitas.csv'))
else:
    print('Penyapuan belum selesai; hasil utama belum dibuat. Periksa checkpoint per kasus.')


## 6. Hitung penimbangan anisotropi

Tahap ini membutuhkan keluaran LBM yang telah selesai. Ia menghitung rasio anisotropi, selisih permeabilitas maksimum-minimum, dan jumlah simpangan baku pembanding.

In [ ]:
!python program/inti/rev_anisotropi.py --stage anisotropy

In [ ]:
from IPython.display import display, Image
display(pd.read_csv('hasil/penimbangan_anisotropi.csv'))
if Path('hasil/perkiraan_REV_permeabilitas.json').exists():
    print(Path('hasil/perkiraan_REV_permeabilitas.json').read_text())
if Path('gambar/cv_tiga_sifat.png').exists():
    display(Image('gambar/cv_tiga_sifat.png'))


## Catatan analisis saya

Isi bagian ini setelah membaca keluaran. Bagian ini sengaja berupa catatan kerja, supaya alasan di balik angka tidak hilang ketika laporan disusun. Jangan menulis kesimpulan yang belum didukung tabel atau grafik.

- Pola yang paling terlihat pada kurva Cv(L):
- Ukuran yang pertama melewati batas 5% untuk setiap sifat:
- Apakah kemiringannya dekat dengan −3/2? Jika berbeda, apa kemungkinan penyebabnya?
- Berapa subvolume yang tidak menembus, belum konvergen, atau gagal pemeriksaan fisis?
- Apakah selisih antararah pada ukuran terbesar lebih dari 2σ?
- Apakah fluks antarlapisan sudah seragam?
- Satu keterbatasan yang paling memengaruhi kesimpulan saya:


**Pemeriksaan terakhir sebelum menarik simpulan:** pastikan jumlah kasus yang lolos pemeriksaan fisis tidak nol, Cv permeabilitas dihitung dari rerata geometrik X–Y–Z per subvolume, dan selisih antararah dibandingkan dengan 2σ pada ukuran L yang sama. Nilai domain penuh dari CSV dosen hanya menjadi titik acuan L=256, bukan pengganti hasil subvolume.


## 7. Catatan analisis dan unduh keluaran

Sebelum menyusun laporan, saya memeriksa kembali CSV dan JSON di folder `hasil/`. Saya menyimpan kasus yang tidak menembus, belum konvergen, atau gagal pemeriksaan fisis. Angka yang masuk grafik dan kesimpulan harus bisa ditelusuri ke baris keluaran program.

Citra pada bundel ini sudah biner. Karena itu, saya tidak membuat rentang ambang grayscale buatan. Ketidaksesuaian antara ketentuan umum ketidakpastian dan batas khusus Bab 6 saya catat untuk dikonfirmasi kepada dosen.


In [ ]:
# Setelah semua hasil diperiksa, arsipkan folder hasil beserta program dan catatan.
import shutil
from google.colab import files
old_archive = ROOT_DRIVE / 'REV-Anisotropi-Kirby-hasil.zip'
if old_archive.exists(): old_archive.unlink()
zip_path = shutil.make_archive('/content/REV-Anisotropi-Kirby-hasil', 'zip', ROOT_DRIVE)
shutil.copy2(zip_path, ROOT_DRIVE / 'REV-Anisotropi-Kirby-hasil.zip')
files.download(zip_path)
